# BirdCLEF 2026 - Experimentation Notebook

Complete notebook for training, validation, and experimentation with the BirdCLEF 2026 solution.

## 1. Setup and Imports

In [ ]:
import os
import sys
import yaml
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, WeightedRandomSampler
from torch.optim.lr_scheduler import CosineAnnealingLR

from tqdm import tqdm
import warnings
warnings.filterwarnings('ignore')

# Set random seeds
np.random.seed(42)
torch.manual_seed(42)
if torch.cuda.is_available():
    torch.cuda.manual_seed(42)

print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

## 2. Load Configuration

In [ ]:
# Load configuration
with open('config.yaml', 'r') as f:
    config = yaml.safe_load(f)

# Set device
if config['training']['device'] == 'auto':
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
else:
    device = torch.device(config['training']['device'])

print(f"Using device: {device}")
print(f"\nConfiguration:")
print(yaml.dump(config, default_flow_style=False))

## 3. Import Project Modules

In [ ]:
# Add project to path
sys.path.insert(0, os.path.abspath('.'))

from src.model import BirdCLEFModel
from src.dataset import BirdCLEFDataset, BirdCLEFCombinedDataset
from src.inference import OptimizedInference

## 4. Load Datasets

In [ ]:
# Load training dataset
train_dataset = BirdCLEFDataset(
    csv_path=config['dataset']['train_csv'],
    audio_dir=config['dataset']['audio_dir'],
    taxonomy_path=config['dataset']['taxonomy_csv'],
    sr=config['dataset']['sr'],
    duration=config['dataset']['duration'],
    n_mels=config['dataset']['n_mels'],
    augment=True,
    split='train',
    val_split=config['dataset']['val_split']
)

# Load validation dataset
val_dataset = BirdCLEFDataset(
    csv_path=config['dataset']['train_csv'],
    audio_dir=config['dataset']['audio_dir'],
    taxonomy_path=config['dataset']['taxonomy_csv'],
    sr=config['dataset']['sr'],
    duration=config['dataset']['duration'],
    n_mels=config['dataset']['n_mels'],
    augment=False,
    split='val',
    val_split=config['dataset']['val_split']
)

print(f"Training samples: {len(train_dataset)}")
print(f"Validation samples: {len(val_dataset)}")
print(f"Number of classes: {train_dataset.num_classes}")

## 5. Create DataLoaders with Class Balancing

In [ ]:
# Get class weights for balancing
print("Computing class weights...")
weights = train_dataset.get_class_weights()

# Create weighted sampler
sampler = WeightedRandomSampler(
    weights=weights,
    num_samples=len(train_dataset),
    replacement=True
)

# Create data loaders
train_loader = DataLoader(
    train_dataset,
    batch_size=config['training']['batch_size'],
    sampler=sampler,
    num_workers=config['training']['num_workers'],
    pin_memory=config['training']['pin_memory']
)

val_loader = DataLoader(
    val_dataset,
    batch_size=config['training']['batch_size'],
    shuffle=False,
    num_workers=config['training']['num_workers'],
    pin_memory=config['training']['pin_memory']
)

print(f"Train batches: {len(train_loader)}")
print(f"Val batches: {len(val_loader)}")

## 6. Visualize Dataset Samples

In [ ]:
# Visualize sample spectrograms
fig, axes = plt.subplots(3, 3, figsize=(15, 12))
axes = axes.flatten()

for idx in range(9):
    spectrogram, target = train_dataset[idx]
    
    # Plot spectrogram
    im = axes[idx].imshow(spectrogram.squeeze().numpy(), aspect='auto', origin='lower')
    axes[idx].set_title(f'Sample {idx}\nPositive labels: {target.nonzero(as_tuple=True)[0].tolist()}')
    axes[idx].set_xlabel('Time')
    axes[idx].set_ylabel('Mel Frequency')
    plt.colorbar(im, ax=axes[idx])

plt.tight_layout()
plt.show()

## 7. Initialize Model, Loss, and Optimizer

In [ ]:
# Create model
model = BirdCLEFModel(
    num_classes=config['dataset']['num_classes'],
    pretrained=config['model']['pretrained']
).to(device)

# Loss function
criterion = nn.BCEWithLogitsLoss()

# Optimizer
optimizer = optim.AdamW(
    model.parameters(),
    lr=config['training']['learning_rate'],
    weight_decay=config['training']['weight_decay']
)

# Scheduler
scheduler = CosineAnnealingLR(
    optimizer,
    T_max=config['training']['num_epochs'],
    eta_min=1e-6
)

print(f"Model: {model.__class__.__name__}")
print(f"Parameters: {sum(p.numel() for p in model.parameters()) / 1e6:.2f}M")
print(f"Loss: {criterion.__class__.__name__}")
print(f"Optimizer: {optimizer.__class__.__name__}")

## 8. Training Loop

In [ ]:
def train_epoch(model, train_loader, optimizer, criterion, device):
    """Train for one epoch."""
    model.train()
    total_loss = 0.0
    
    pbar = tqdm(train_loader, desc="Training")
    for spectrograms, labels in pbar:
        spectrograms = spectrograms.to(device)
        labels = labels.to(device)
        
        optimizer.zero_grad()
        
        # Forward pass
        logits = model(spectrograms)
        loss = criterion(logits, labels)
        
        # Backward pass
        loss.backward()
        optimizer.step()
        
        total_loss += loss.item()
        pbar.set_postfix({'loss': f'{loss.item():.4f}'})
    
    return total_loss / len(train_loader)

def validate(model, val_loader, criterion, device):
    """Validate the model."""
    model.eval()
    total_loss = 0.0
    
    with torch.no_grad():
        pbar = tqdm(val_loader, desc="Validating")
        for spectrograms, labels in pbar:
            spectrograms = spectrograms.to(device)
            labels = labels.to(device)
            
            logits = model(spectrograms)
            loss = criterion(logits, labels)
            
            total_loss += loss.item()
            pbar.set_postfix({'loss': f'{loss.item():.4f}'})
    
    return total_loss / len(val_loader)

print("Training functions defined.")

## 9. Run Training

In [ ]:
# Create checkpoint directory
os.makedirs(config['checkpoint']['save_dir'], exist_ok=True)

best_val_loss = float('inf')
train_losses = []
val_losses = []

# Training loop
for epoch in range(config['training']['num_epochs']):
    print(f"\nEpoch {epoch + 1}/{config['training']['num_epochs']}")
    
    # Train
    train_loss = train_epoch(model, train_loader, optimizer, criterion, device)
    train_losses.append(train_loss)
    print(f"Train Loss: {train_loss:.4f}")
    
    # Validate
    val_loss = validate(model, val_loader, criterion, device)
    val_losses.append(val_loss)
    print(f"Val Loss: {val_loss:.4f}")
    
    # Update scheduler
    scheduler.step()
    
    # Save best model
    if val_loss < best_val_loss:
        best_val_loss = val_loss
        torch.save(model.state_dict(), os.path.join(config['checkpoint']['save_dir'], 'best_model.pt'))
        print("✓ Best model saved!")

print("\nTraining complete!")

## 10. Plot Training Curves

In [ ]:
plt.figure(figsize=(10, 6))
plt.plot(train_losses, label='Train Loss', marker='o')
plt.plot(val_losses, label='Val Loss', marker='s')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.title('Training and Validation Loss')
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

print(f"Best validation loss: {min(val_losses):.4f}")

## 11. Model Evaluation

In [ ]:
# Load best model
model.load_state_dict(torch.load(os.path.join(config['checkpoint']['save_dir'], 'best_model.pt')))
model.eval()

# Get validation predictions
all_preds = []
all_targets = []

with torch.no_grad():
    for spectrograms, labels in tqdm(val_loader, desc="Evaluating"):
        spectrograms = spectrograms.to(device)
        labels = labels.to(device)
        
        logits = model(spectrograms)
        probs = torch.sigmoid(logits)
        
        all_preds.append(probs.cpu().numpy())
        all_targets.append(labels.cpu().numpy())

all_preds = np.concatenate(all_preds, axis=0)
all_targets = np.concatenate(all_targets, axis=0)

print(f"Predictions shape: {all_preds.shape}")
print(f"Targets shape: {all_targets.shape}")

## 12. Save Final Model

In [ ]:
# Save final model
torch.save(model.state_dict(), 'model.pt')
print("Model saved to model.pt")

# Get model info
total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)

print(f"Total parameters: {total_params / 1e6:.2f}M")
print(f"Trainable parameters: {trainable_params / 1e6:.2f}M")

## 13. Test Inference

In [ ]:
# Load test sample
test_spec, test_target = val_dataset[0]

# Run inference
with torch.no_grad():
    logits = model(test_spec.unsqueeze(0).to(device))
    probs = torch.sigmoid(logits).cpu().numpy()[0]

# Get top predictions
top_indices = np.argsort(probs)[-5:][::-1]

print("Top 5 predicted species:")
for i, idx in enumerate(top_indices):
    print(f"{i+1}. Species {idx}: {probs[idx]:.4f}")

## 14. Experimentation Space

Use this section to experiment with different configurations, augmentations, or model variations.

In [ ]:
# Add your experiments here
# For example:
# - Try different learning rates
# - Test different augmentation strategies
# - Analyze misclassified samples
# - Visualize attention maps
# - Test ensemble approaches

print("Ready for experimentation!")